In [21]:
import threading
import numpy as np
import time
from PIL import Image
from constants import PGM
import multiprocessing
from concurrent.futures import ThreadPoolExecutor

In [22]:
def image_negative_worker(start, end,result_lock, new_image, image):
    """Generate a negative for the image chunk of the image"""
    image_chunk = image[start:end].copy()
    new_image_chunk = 255 - image_chunk
    # Need lock when updating shared result
    with result_lock:
        new_image[start:end] += new_image_chunk
        
def threshold_with_slicing_worker(start, end,result_lock, new_image, image, limits, maxv):
    """Generate a negative for the image chunk of the image"""
    a, b = limits
    image_chunk = image[start:end].copy()
    mask = (image_chunk > a) & (image_chunk < b)
    new_image_chunk = np.where(mask, maxv, image_chunk)
    
    # Need lock when updating shared result
    with result_lock:
        new_image[start:end] = new_image_chunk

In [23]:
def calculate_image_threads(height, min_rows_per_thread=50, max_threads=16):
    """
    Calculate threads for image processing

    Args:
        height: Image height in pixels
        min_rows_per_thread: Minimum rows to avoid overhead
        max_threads: Upper limit
    """
    cpu_cores = multiprocessing.cpu_count()

    # Maximum threads possible based on rows
    max_threads_by_rows = max(1, height // min_rows_per_thread)

    # Optimal threads (can't have more threads than rows)
    optimal = min(cpu_cores, max_threads_by_rows, max_threads)

    # Keep at least one CPU core free when all cores would be used
    if optimal == cpu_cores and optimal > 1:
        optimal -= 1

    # Ensure each thread gets reasonable work
    if optimal > 1:
        rows_per_thread = height // optimal
        if rows_per_thread < min_rows_per_thread:
            optimal = max(1, height // min_rows_per_thread)
            if optimal == cpu_cores and optimal > 1:
                optimal -= 1

    return optimal

In [24]:
def get_image_thread_results(image_path):
    with Image.open(image_path) as img:
        arr = np.array(img)
        data = arr.tobytes()
        metadata = PGM(img.height, img.width, int(arr.max()), data)
    threads = calculate_image_threads(metadata.height)
    return {
        "metadata": metadata,
        "threads": threads,
    }


def get_image_slice_values(image_path, colored=False):
    results = get_image_thread_results(image_path)
    metadata = results["metadata"]

    width, height = metadata.width, metadata.height
    if colored:
        image = np.frombuffer(metadata.data, dtype=np.uint8).reshape(height, width, 3)
    else:
        image = np.frombuffer(metadata.data, dtype=np.uint8).reshape(height, width)

    num_threads = results["threads"]
    rows_per_thread = height // num_threads

    slices = []
    for i in range(num_threads):
        start = i * rows_per_thread
        end = (i + 1) * rows_per_thread if i < num_threads - 1 else height
        slices.append([start, end]) # slices

    return {
        "metadata": metadata,
        "image": image,
        "slices": slices,
    }

In [25]:
def generate_threads(image, slices, worker, args=None):
    if args is None:
        args = []

    new_image = np.zeros_like(image)
    start_time = time.time()
    lock = threading.Lock()

    with ThreadPoolExecutor(max_workers=len(slices)) as executor:
        futures = [
            executor.submit(worker, start, end, lock, new_image, image, *args)
            for start, end in slices
        ]

        for future in futures:
            future.result()

    elapsed = time.time() - start_time
    return new_image, elapsed

def process_image_with_worker(
    worker,
    image_path,
    output_path,
    worker_args=None,
    colored=False,
):
    image_values = get_image_slice_values(image_path=image_path, colored=colored)
    metadata = image_values["metadata"]
    image = image_values["image"]
    slices = image_values["slices"]

    args = worker_args(metadata) if callable(worker_args) else (worker_args or [])

    new_image, elapsed = generate_threads(
        image,
        slices,
        worker,
        args,
    )

    altered_bytes = new_image.tobytes()
    if ".pgm" in output_path:
        header = f"P5\n{metadata.width} {metadata.height}\n255\n".encode()
    else:
        header = f"P6\n{metadata.width} {metadata.height}\n255\n".encode()

    with open(output_path, "wb") as f:
        f.write(header)
        f.write(altered_bytes)

    return elapsed

if __name__ == "__main__":
    elapsed = process_image_with_worker(
        threshold_with_slicing_worker,
        image_path="vazo.pgm",
        output_path="output.pgm",
        worker_args=lambda metadata: [(175, 255), metadata.maxv],
        colored=False,
    )
    print(f"Time taken: {elapsed:.4f} seconds")
    
    elapsed = process_image_with_worker(
        image_negative_worker,
        image_path="horse.ppm",
        output_path="output.ppm",
        colored=True,
    )
    print(f"Time taken: {elapsed:.4f} seconds")

Time taken: 0.0016 seconds
Time taken: 0.0404 seconds
